In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter
from tqdm.notebook import tqdm
import copy
from pathlib import Path
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs, prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import build_weekly_calendar_from_panel
from pd_estim_A.models.nig.nig_em_paper import (
    EM_algo,
    update_theta,
    invert_nig_call_price,
    compute_pd_physical,
    compute_pd_risk_neutral,
)

In [2]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931

In [3]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
nig = nig_df.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"]  = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

nig_df = merged_cds.reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 76508
date range: 2012-01-03 00:00:00 → 2025-12-19 00:00:00


In [4]:
# Single-firm pilot selection
TEST_GVKEY = "100080"   # Bayer example; set to None to auto-pick first eligible firm
EVAL_YEAR = 2014        # first full year with a 2Y burn-in when panel starts in 2012
TRAIN_YEARS = 2
WEEK_ENDING = "W-FRI"


def prepare_one_firm_panel(df: pd.DataFrame, gvkey: str) -> pd.DataFrame:
    g = (
        df.loc[df["gvkey"].astype(str) == str(gvkey)]
          .copy()
          .sort_values("date")
          .reset_index(drop=True)
    )
    g["gvkey"] = g["gvkey"].astype(str)
    g["date"] = pd.to_datetime(g["date"])

    num_cols = [c for c in ["E", "L", "r", "cds"] if c in g.columns]
    for c in num_cols:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    g = (
        g.dropna(subset=["date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values("date")
         .groupby("date", as_index=False)
         .last()
         .reset_index(drop=True)
    )
    return g


def make_quarterly_nig_pilot_windows(
    g_onefirm: pd.DataFrame,
    *,
    eval_year: int,
    train_years: int = 2,
    week_ending: str = "W-FRI",
) -> list[dict]:
    g = g_onefirm.sort_values("date").copy()
    weekly_all = pd.DatetimeIndex(build_weekly_calendar_from_panel(g, week_ending=week_ending))

    eval_start = pd.Timestamp(f"{eval_year}-01-01")
    eval_end   = pd.Timestamp(f"{eval_year}-12-31")

    weekly_eval = weekly_all[(weekly_all >= eval_start) & (weekly_all <= eval_end)]
    if len(weekly_eval) == 0:
        raise ValueError(f"No weekly dates found for eval_year={eval_year}.")

    anchors_ser = (
        pd.Series(weekly_eval, index=weekly_eval)
          .groupby(weekly_eval.to_period("Q"))
          .min()
    )

    if len(anchors_ser) != 4:
        raise ValueError(
            f"Expected 4 quarter anchors in eval_year={eval_year}, got {len(anchors_ser)}."
        )

    anchors = pd.DatetimeIndex(anchors_ser.values)
    windows = []

    for i, anchor in enumerate(anchors):
        train_start = anchor - pd.DateOffset(years=train_years) + pd.Timedelta(days=1)
        train_end = anchor

        if i < len(anchors) - 1:
            next_anchor = anchors[i + 1]
            score_dates = weekly_eval[(weekly_eval >= anchor) & (weekly_eval < next_anchor)]
        else:
            next_anchor = pd.NaT
            score_dates = weekly_eval[(weekly_eval >= anchor) & (weekly_eval <= eval_end)]

        windows.append({
            "quarter_no": i + 1,
            "anchor_date": pd.Timestamp(anchor),
            "next_anchor": pd.Timestamp(next_anchor) if pd.notna(next_anchor) else pd.NaT,
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "score_dates": pd.DatetimeIndex(score_dates),
        })

    return windows


# choose firm
if TEST_GVKEY is None:
    eligible = []
    all_gvkeys = np.sort(nig_df["gvkey"].astype(str).dropna().unique())

    for gv in all_gvkeys:
        try:
            g_try = prepare_one_firm_panel(nig_df, gv)
            w_try = make_quarterly_nig_pilot_windows(
                g_try,
                eval_year=EVAL_YEAR,
                train_years=TRAIN_YEARS,
                week_ending=WEEK_ENDING,
            )
            first_train_start = w_try[0]["train_start"]
            if first_train_start >= g_try["date"].min():
                eligible.append(gv)
        except Exception:
            pass

    if len(eligible) == 0:
        raise ValueError("No eligible firm found for the requested pilot configuration.")

    test_gvkey = eligible[0]
else:
    test_gvkey = str(TEST_GVKEY)

nig_df_onefirm = prepare_one_firm_panel(nig_df, test_gvkey)
pilot_windows = make_quarterly_nig_pilot_windows(
    nig_df_onefirm,
    eval_year=EVAL_YEAR,
    train_years=TRAIN_YEARS,
    week_ending=WEEK_ENDING,
)

company_name = (
    nig_df.loc[nig_df["gvkey"].astype(str) == str(test_gvkey), "company"]
          .dropna()
          .iloc[0]
    if "company" in nig_df.columns and
       (nig_df["gvkey"].astype(str) == str(test_gvkey)).any() and
       nig_df.loc[nig_df["gvkey"].astype(str) == str(test_gvkey), "company"].notna().any()
    else "NA"
)

print("chosen gvkey:", test_gvkey)
print("company:", company_name)
print("rows:", len(nig_df_onefirm))
print("date range:", nig_df_onefirm["date"].min(), "->", nig_df_onefirm["date"].max())
print("evaluation year:", EVAL_YEAR)
print("number of quarter anchors:", len(pilot_windows))
print("quarter anchors:", [w["anchor_date"].date() for w in pilot_windows])

display(nig_df_onefirm.head())

chosen gvkey: 100080
company: BAYER AG
rows: 3644
date range: 2012-01-03 00:00:00 -> 2025-12-19 00:00:00
evaluation year: 2014
number of quarter anchors: 4
quarter anchors: [datetime.date(2014, 1, 3), datetime.date(2014, 4, 4), datetime.date(2014, 7, 4), datetime.date(2014, 10, 3)]


,date,gvkey,E,isin,company,country_iso,r,L,cds
0,2012-01-03,100080,4.268705e+10,DE000BAY0017,BAYER AG,DEU,0.001177,3.254300e+10,NaN
1,2012-01-04,100080,4.258781e+10,DE000BAY0017,BAYER AG,DEU,0.001037,3.254300e+10,NaN
2,2012-01-05,100080,4.322456e+10,DE000BAY0017,BAYER AG,DEU,0.001614,3.254300e+10,NaN
3,2012-01-06,100080,4.281109e+10,DE000BAY0017,BAYER AG,DEU,0.001873,3.254300e+10,NaN
4,2012-01-09,100080,4.238108e+10,DE000BAY0017,BAYER AG,DEU,0.001835,3.254300e+10,NaN


In [8]:
def _nig_params_basic_ok(params: dict) -> bool:
    try:
        alpha = float(params["alpha"])
        beta1 = float(params["beta1"])
        delta = float(params["delta"])
        beta0 = float(params["beta0"])
    except Exception:
        return False

    if not np.isfinite(alpha) or not np.isfinite(beta1) or not np.isfinite(delta) or not np.isfinite(beta0):
        return False
    if alpha <= 0.5:
        return False
    if delta <= 0.0:
        return False
    if abs(beta1) >= alpha:
        return False
    return True


def run_one_firm_nig_pilot(
    g_onefirm: pd.DataFrame,
    *,
    windows: list[dict],
    start_params: dict,
    T_horizon: float = 1.0,
    em_max_iter: int = 10,
    em_min_iter: int = 3,
    em_tol: float = 1e-3,
    min_train_rows: int = 250,
    show_progress: bool = True,
    use_quarterly_warm_start: bool = True,
    retry_cold_if_warm_fails: bool = True,
):
    g = g_onefirm.copy().sort_values("date").reset_index(drop=True)
    g["date"] = pd.to_datetime(g["date"])
    g_idx = g.set_index("date").sort_index()

    firm_meta = {}
    for c in ["gvkey", "company", "isin", "country_iso"]:
        if c in g.columns and g[c].notna().any():
            firm_meta[c] = g[c].dropna().iloc[0]
        else:
            firm_meta[c] = np.nan

    quarter_rows = []
    weekly_rows = []

    total_t0 = perf_counter()

    windows_iter = tqdm(
        windows,
        desc=f"NIG pilot | gvkey={firm_meta['gvkey']}",
        unit="quarter",
        leave=True,
    ) if show_progress else windows

    # cold/base seed
    base_start_params = {
        "alpha": float(start_params["alpha"]),
        "beta1": float(start_params["beta1"]),
        "delta": float(start_params["delta"]),
        "beta0": float(start_params["beta0"]),
    }

    # rolling same-firm warm start state
    current_start_params = copy.deepcopy(base_start_params)

    for w in windows_iter:
        quarter_t0 = perf_counter()

        q_no = int(w["quarter_no"])
        anchor = pd.Timestamp(w["anchor_date"])
        train_start = pd.Timestamp(w["train_start"])
        train_end = pd.Timestamp(w["train_end"])
        score_dates = pd.DatetimeIndex(w["score_dates"])

        train_df = g[(g["date"] >= train_start) & (g["date"] <= train_end)].copy()

        base_row = {
            "gvkey": firm_meta["gvkey"],
            "company": firm_meta["company"],
            "isin": firm_meta["isin"],
            "country_iso": firm_meta["country_iso"],
            "quarter_no": q_no,
            "anchor_date": anchor,
            "train_start": train_start,
            "train_end": train_end,
            "n_train_rows": int(len(train_df)),
        }

        if len(train_df) < min_train_rows:
            quarter_elapsed = perf_counter() - quarter_t0

            quarter_rows.append({
                **base_row,
                "ok": False,
                "msg": f"Too few daily rows in 2Y training window: {len(train_df)}",
                "em_converged": False,
                "em_n_iter": np.nan,
                "alpha": np.nan,
                "beta1": np.nan,
                "delta": np.nan,
                "beta0": np.nan,
                "theta_anchor": np.nan,
                "A_anchor": np.nan,
                "L_anchor": np.nan,
                "r_anchor": np.nan,
                "PD_P_anchor": np.nan,
                "PD_Q_anchor": np.nan,
                "quarter_runtime_sec": quarter_elapsed,
                "start_source": "none",
                "warm_start_used": False,
                "warm_start_retry_to_cold": False,
            })

            if show_progress:
                windows_iter.set_postfix({
                    "q": q_no,
                    "anchor": anchor.date().isoformat(),
                    "status": "too_few_rows",
                    "sec": f"{quarter_elapsed:.1f}",
                })
            continue

        # choose start params for this quarter
        if use_quarterly_warm_start and _nig_params_basic_ok(current_start_params):
            em_start_params = copy.deepcopy(current_start_params)
            start_source = "warm_prev_quarter"
            warm_start_used = True
        else:
            em_start_params = copy.deepcopy(base_start_params)
            start_source = "cold_base"
            warm_start_used = False

        warm_retry_to_cold = False

        try:
            try:
                em_out = EM_algo(
                    E_series=train_df["E"].to_numpy(dtype=float),
                    L_face_series=train_df["L"].to_numpy(dtype=float),
                    rf_series=train_df["r"].to_numpy(dtype=float),
                    dates=train_df["date"].to_numpy(),
                    start_params=em_start_params,
                    start_date=None,
                    end_date=None,
                    max_iter=em_max_iter,
                    min_iter=em_min_iter,
                    tol=em_tol,
                )
            except Exception as e_first:
                # guarded fallback: if warm start fails, retry from cold/base seed
                if warm_start_used and retry_cold_if_warm_fails:
                    warm_retry_to_cold = True
                    em_out = EM_algo(
                        E_series=train_df["E"].to_numpy(dtype=float),
                        L_face_series=train_df["L"].to_numpy(dtype=float),
                        rf_series=train_df["r"].to_numpy(dtype=float),
                        dates=train_df["date"].to_numpy(),
                        start_params=base_start_params,
                        start_date=None,
                        end_date=None,
                        max_iter=em_max_iter,
                        min_iter=em_min_iter,
                        tol=em_tol,
                    )
                    start_source = "warm_failed_then_cold"
                    em_start_params = copy.deepcopy(base_start_params)
                else:
                    raise e_first

            params = dict(em_out["params"])
            A_anchor = float(em_out["A_win"][-1])
            theta_anchor = float(em_out["theta_win"][-1])

            row_anchor = g_idx.loc[anchor]
            L_anchor = float(row_anchor["L"])
            r_anchor = float(row_anchor["r"])
            E_anchor = float(row_anchor["E"])
            cds_anchor = (
                float(row_anchor["cds"])
                if "cds" in row_anchor.index and pd.notna(row_anchor["cds"])
                else np.nan
            )

            params_anchor = dict(params)
            params_anchor["theta"] = theta_anchor

            PD_P_anchor = float(
                compute_pd_physical(
                    A0=A_anchor,
                    L=L_anchor,
                    T=T_horizon,
                    params=params,
                )
            )
            PD_Q_anchor = float(
                compute_pd_risk_neutral(
                    A0=A_anchor,
                    L=L_anchor,
                    T=T_horizon,
                    params=params_anchor,
                )
            )

            quarter_rows.append({
                **base_row,
                "ok": True,
                "msg": "ok",
                "em_converged": bool(em_out["converged"]),
                "em_n_iter": int(em_out["n_iter"]),
                "alpha": float(params["alpha"]),
                "beta1": float(params["beta1"]),
                "delta": float(params["delta"]),
                "beta0": float(params["beta0"]),
                "theta_anchor": theta_anchor,
                "A_anchor": A_anchor,
                "L_anchor": L_anchor,
                "r_anchor": r_anchor,
                "E_anchor": E_anchor,
                "cds_anchor": cds_anchor,
                "PD_P_anchor": PD_P_anchor,
                "PD_Q_anchor": PD_Q_anchor,
                "quarter_runtime_sec": np.nan,  # filled below
                "start_source": start_source,
                "warm_start_used": bool(warm_start_used),
                "warm_start_retry_to_cold": bool(warm_retry_to_cold),
            })

            # if successful, update same-firm rolling warm start for next quarter
            if use_quarterly_warm_start and _nig_params_basic_ok(params):
                current_start_params = {
                    "alpha": float(params["alpha"]),
                    "beta1": float(params["beta1"]),
                    "delta": float(params["delta"]),
                    "beta0": float(params["beta0"]),
                }
            else:
                current_start_params = copy.deepcopy(base_start_params)

            prev_score_A = None

            for d in score_dates:
                row_t = g_idx.loc[d]

                E_t = float(row_t["E"])
                L_t = float(row_t["L"])
                r_t = float(row_t["r"])
                cds_t = (
                    float(row_t["cds"])
                    if "cds" in row_t.index and pd.notna(row_t["cds"])
                    else np.nan
                )

                if pd.Timestamp(d) == anchor:
                    A_t = A_anchor
                    theta_t = theta_anchor
                    source = "anchor_em"
                else:
                    theta_t = float(update_theta(params, r_t))
                    params_t_for_inversion = dict(params)
                    params_t_for_inversion["theta"] = theta_t

                    A_t = float(
                        invert_nig_call_price(
                            E_obs=E_t,
                            L_face=L_t,
                            r=r_t,
                            T=T_horizon,
                            params=params_t_for_inversion,
                            discounting="continuous",
                        )
                    )
                    source = "weekly_rescore"

                dlogA = np.nan
                if (
                    prev_score_A is not None
                    and np.isfinite(prev_score_A)
                    and prev_score_A > 0
                    and np.isfinite(A_t)
                    and A_t > 0
                ):
                    dlogA = float(np.log(A_t / prev_score_A))

                prev_score_A = A_t

                params_t = dict(params)
                params_t["theta"] = theta_t

                PD_P_t = float(
                    compute_pd_physical(
                        A0=A_t,
                        L=L_t,
                        T=T_horizon,
                        params=params,
                    )
                )
                PD_Q_t = float(
                    compute_pd_risk_neutral(
                        A0=A_t,
                        L=L_t,
                        T=T_horizon,
                        params=params_t,
                    )
                )

                weekly_rows.append({
                    "gvkey": firm_meta["gvkey"],
                    "company": firm_meta["company"],
                    "isin": firm_meta["isin"],
                    "country_iso": firm_meta["country_iso"],
                    "quarter_no": q_no,
                    "anchor_date": anchor,
                    "date": pd.Timestamp(d),
                    "source": source,
                    "train_start": train_start,
                    "train_end": train_end,
                    "alpha": float(params["alpha"]),
                    "beta1": float(params["beta1"]),
                    "delta": float(params["delta"]),
                    "beta0": float(params["beta0"]),
                    "theta": float(theta_t),
                    "A_hat": float(A_t),
                    "dlogA": dlogA,
                    "E": E_t,
                    "L": L_t,
                    "r": r_t,
                    "cds": cds_t,
                    "PD_P_1y": PD_P_t,
                    "PD_Q_1y": PD_Q_t,
                    "is_anchor_date": pd.Timestamp(d) == anchor,
                    "refit_id": f"{firm_meta['gvkey']}_{anchor:%Y-%m-%d}",
                    "start_source": start_source,
                    "warm_start_used": bool(warm_start_used),
                    "warm_start_retry_to_cold": bool(warm_retry_to_cold),
                })

            quarter_elapsed = perf_counter() - quarter_t0
            quarter_rows[-1]["quarter_runtime_sec"] = quarter_elapsed

            if show_progress:
                windows_iter.set_postfix({
                    "q": q_no,
                    "anchor": anchor.date().isoformat(),
                    "status": "ok",
                    "start": start_source,
                    "em_iter": int(em_out["n_iter"]),
                    "sec": f"{quarter_elapsed:.1f}",
                })

        except Exception as e:
            quarter_elapsed = perf_counter() - quarter_t0

            quarter_rows.append({
                **base_row,
                "ok": False,
                "msg": str(e),
                "em_converged": False,
                "em_n_iter": np.nan,
                "alpha": np.nan,
                "beta1": np.nan,
                "delta": np.nan,
                "beta0": np.nan,
                "theta_anchor": np.nan,
                "A_anchor": np.nan,
                "L_anchor": np.nan,
                "r_anchor": np.nan,
                "PD_P_anchor": np.nan,
                "PD_Q_anchor": np.nan,
                "quarter_runtime_sec": quarter_elapsed,
                "start_source": start_source,
                "warm_start_used": bool(warm_start_used),
                "warm_start_retry_to_cold": bool(warm_retry_to_cold),
            })

            # reset to base seed after failure
            current_start_params = copy.deepcopy(base_start_params)

            if show_progress:
                windows_iter.set_postfix({
                    "q": q_no,
                    "anchor": anchor.date().isoformat(),
                    "status": "fail",
                    "start": start_source,
                    "sec": f"{quarter_elapsed:.1f}",
                })
            continue

    total_elapsed = perf_counter() - total_t0

    quarter_df = pd.DataFrame(quarter_rows).sort_values(["anchor_date"]).reset_index(drop=True)
    weekly_df = pd.DataFrame(weekly_rows).sort_values(["date"]).reset_index(drop=True)

    return quarter_df, weekly_df, total_elapsed

In [9]:
# Run the one-firm pilot with guarded quarterly warm starts
START_PARAMS = {
    "alpha": 10.0,
    "beta1": 0.0,
    "delta": 1.0,
    "beta0": 0.0,
}

pilot_quarter_nig_df, pilot_weekly_nig_df, total_elapsed = run_one_firm_nig_pilot(
    nig_df_onefirm,
    windows=pilot_windows,
    start_params=START_PARAMS,
    T_horizon=1.0,
    em_max_iter=10,
    em_min_iter=3,
    em_tol=1e-3,
    min_train_rows=250,
    show_progress=True,
    use_quarterly_warm_start=True,
    retry_cold_if_warm_fails=True,
)

print(f"Total runtime: {total_elapsed:.2f} seconds")
print("pilot_quarter_nig_df shape:", pilot_quarter_nig_df.shape)
print("pilot_weekly_nig_df shape:", pilot_weekly_nig_df.shape)

display(
    pilot_quarter_nig_df[
        [
            "gvkey", "company", "quarter_no", "anchor_date",
            "start_source", "warm_start_used", "warm_start_retry_to_cold",
            "em_converged", "em_n_iter", "quarter_runtime_sec",
            "alpha", "beta1", "delta", "beta0",
            "A_anchor", "L_anchor", "r_anchor",
            "PD_Q_anchor", "PD_P_anchor", "cds_anchor"
        ]
    ]
)

display(
    pilot_weekly_nig_df[
        [
            "gvkey", "company", "quarter_no", "anchor_date", "date", "source",
            "start_source", "warm_start_used", "warm_start_retry_to_cold",
            "alpha", "beta1", "delta", "beta0", "theta",
            "A_hat", "L", "r", "PD_Q_1y", "PD_P_1y", "cds"
        ]
    ].head(40)
)

NIG pilot | gvkey=100080: 100%|██████████| 4/4 [04:26<00:00, 66.60s/quarter, q=4, anchor=2014-10-03, status=ok, start=warm_prev_quarter, em_iter=4, sec=58.9]

Total runtime: 266.42 seconds
pilot_quarter_nig_df shape: (4, 29)
pilot_weekly_nig_df shape: (52, 28)


,gvkey,company,quarter_no,anchor_date,start_source,warm_start_used,warm_start_retry_to_cold,em_converged,em_n_iter,quarter_runtime_sec,alpha,beta1,delta,beta0,A_anchor,L_anchor,r_anchor,PD_Q_anchor,PD_P_anchor,cds_anchor
0,100080,BAYER AG,1,2014-01-03,warm_prev_quarter,True,False,True,3,44.290455,150.291186,7.546631,3.003749,0.056639,1.160909e+11,3.276700e+10,0.000991,2.136974e-18,6.918153e-25,8.05
1,100080,BAYER AG,2,2014-04-04,warm_prev_quarter,True,False,True,3,82.422219,116.809212,3.671908,2.666617,0.120640,1.122948e+11,3.046400e+10,0.001217,8.172367e-17,1.310141e-22,6.48
2,100080,BAYER AG,3,2014-07-04,warm_prev_quarter,True,False,True,3,80.749697,122.654073,13.692812,2.566542,-0.094174,1.173817e+11,3.046400e+10,-0.000181,7.839417e-20,5.732767e-26,6.42
3,100080,BAYER AG,4,2014-10-03,warm_prev_quarter,True,False,True,4,58.935428,116.490598,8.386744,2.556871,-0.024217,1.202792e+11,3.046400e+10,-0.000829,2.348812e-19,2.499482e-24,7.21


,gvkey,company,quarter_no,anchor_date,date,source,start_source,warm_start_used,warm_start_retry_to_cold,alpha,beta1,delta,beta0,theta,A_hat,L,r,PD_Q_1y,PD_P_1y,cds
0,100080,BAYER AG,1,2014-01-03,2014-01-03,anchor_em,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.830443,1.160909e+11,3.276700e+10,0.000991,2.136974e-18,6.918153e-25,8.05
1,100080,BAYER AG,1,2014-01-03,2014-01-10,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.831850,1.147356e+11,3.276700e+10,0.000963,4.222883e-18,1.547862e-24,8.11
2,100080,BAYER AG,1,2014-01-03,2014-01-17,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.830198,1.165042e+11,3.276700e+10,0.000996,1.737499e-18,5.416193e-25,7.45
3,100080,BAYER AG,1,2014-01-03,2014-01-24,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.840309,1.147081e+11,3.276700e+10,0.000794,4.327619e-18,1.573469e-24,8.70
4,100080,BAYER AG,1,2014-01-03,2014-01-31,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.861442,1.137048e+11,3.276700e+10,0.000371,7.360469e-18,2.864042e-24,8.63
5,100080,BAYER AG,1,2014-01-03,2014-02-07,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.849540,1.116875e+11,3.276700e+10,0.000609,2.009029e-17,9.616934e-24,6.79
6,100080,BAYER AG,1,2014-01-03,2014-02-14,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.851958,1.163531e+11,3.276700e+10,0.000561,1.926533e-18,5.923049e-25,6.80
7,100080,BAYER AG,1,2014-01-03,2014-02-21,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.848033,1.165986e+11,3.276700e+10,0.000639,1.695672e-18,5.122094e-25,6.62
8,100080,BAYER AG,1,2014-01-03,2014-02-28,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.836402,1.178314e+11,3.276700e+10,0.000872,9.036341e-19,2.474663e-25,6.54
9,100080,BAYER AG,1,2014-01-03,2014-03-07,weekly_rescore,warm_prev_quarter,True,False,150.291186,7.546631,3.003749,0.056639,-10.818726,1.104008e+11,3.046400e+10,0.001225,5.604032e-19,1.445052e-25,6.59


In [ ]:
# ADD THIS IMPORT
from pd_estim_A.models.nig.nig_em_paper import nig_call_price
# SANITY-CHECK HELPERS

def rough_pd_from_cds_1y(cds_bps: float, recovery: float = 0.40) -> float:
    """
    Rough 1Y CDS-implied PD sanity check under flat hazard:
      lambda ~ s / (1 - R),  PD(1Y) = 1 - exp(-lambda)
    where s is CDS spread in decimal per year.
    This is only a rough comparison metric, not a full CDS bootstrap.
    """
    if not np.isfinite(cds_bps) or cds_bps < 0 or recovery >= 1:
        return np.nan
    s = cds_bps / 10000.0
    lam = s / max(1e-12, (1.0 - recovery))
    return float(1.0 - np.exp(-lam))


def run_sanity_checks_nig_pilot(
    quarter_df: pd.DataFrame,
    weekly_df: pd.DataFrame,
    onefirm_daily_df: pd.DataFrame,
    *,
    week_ending: str = "W-FRI",
    recovery_for_cds: float = 0.40,
):
    q = quarter_df.copy()
    w = weekly_df.copy()
    g = onefirm_daily_df.copy()

    q["anchor_date"] = pd.to_datetime(q["anchor_date"])
    w["anchor_date"] = pd.to_datetime(w["anchor_date"])
    w["date"] = pd.to_datetime(w["date"])
    g["date"] = pd.to_datetime(g["date"])

    # -------------------------
    # 1) CALENDAR / PANEL SHAPE
    # -------------------------
    g = g.sort_values("date").copy()
    g["week"] = g["date"].dt.to_period(week_ending)
    week_last_trade = g.groupby("week")["date"].max()

    w["week"] = w["date"].dt.to_period(week_ending)
    w["expected_last_trade"] = w["week"].map(week_last_trade)
    w["is_last_trade_of_week"] = w["date"] == w["expected_last_trade"]

    shape_checks = pd.Series({
        "n_quarters": len(q),
        "n_weekly_rows": len(w),
        "n_unique_weekly_dates": w["date"].nunique(),
        "n_duplicate_quarter_date_rows": int(w.duplicated(["quarter_no", "date"]).sum()),
        "all_weekly_dates_are_last_trading_day": bool(w["is_last_trade_of_week"].all()),
        "n_anchor_rows": int((w["source"] == "anchor_em").sum()),
        "one_anchor_per_quarter": bool((w.groupby("quarter_no")["source"].apply(lambda s: (s == "anchor_em").sum()) == 1).all()),
    })


    # 2) PARAMETER / THETA / PRICING FEASIBILITY

    w["param_basic_ok"] = (
        np.isfinite(w["alpha"]) &
        np.isfinite(w["beta1"]) &
        np.isfinite(w["delta"]) &
        np.isfinite(w["beta0"]) &
        (w["alpha"] > 0.5) &
        (w["delta"] > 0.0) &
        (np.abs(w["beta1"]) < w["alpha"])
    )

    w["pricing_feasible_ok"] = (
        np.isfinite(w["theta"]) &
        (np.abs(w["beta1"] + w["theta"]) < w["alpha"]) &
        (np.abs(w["beta1"] + w["theta"] + 1.0) < w["alpha"])
    )

    feasibility_checks = pd.Series({
        "all_basic_params_feasible": bool(w["param_basic_ok"].all()),
        "all_weekly_pricing_feasible": bool(w["pricing_feasible_ok"].all()),
        "min_alpha_minus_abs_beta_theta": float((w["alpha"] - np.abs(w["beta1"] + w["theta"])).min()),
        "min_alpha_minus_abs_beta_theta_plus1": float((w["alpha"] - np.abs(w["beta1"] + w["theta"] + 1.0)).min()),
    })


    # 3) REPRICING CHECK: does A_hat price E back?

    def _reprice_equity(row):
        params = {
            "alpha": float(row["alpha"]),
            "beta1": float(row["beta1"]),
            "delta": float(row["delta"]),
            "beta0": float(row["beta0"]),
            "theta": float(row["theta"]),
        }
        return float(
            nig_call_price(
                A=float(row["A_hat"]),
                L_face=float(row["L"]),
                r=float(row["r"]),
                T=1.0,
                params=params,
                discounting="continuous",
            )
        )

    w["E_model_from_Ahat"] = w.apply(_reprice_equity, axis=1)
    w["repricing_abs_err"] = np.abs(w["E_model_from_Ahat"] - w["E"])
    w["repricing_rel_err"] = w["repricing_abs_err"] / np.maximum(np.abs(w["E"]), 1e-12)

    repricing_checks = pd.Series({
        "all_Ahat_positive": bool((w["A_hat"] > 0).all()),
        "all_Ahat_ge_E": bool((w["A_hat"] >= w["E"]).all()),
        "max_abs_repricing_err": float(w["repricing_abs_err"].max()),
        "median_abs_repricing_err": float(w["repricing_abs_err"].median()),
        "max_rel_repricing_err": float(w["repricing_rel_err"].max()),
        "median_rel_repricing_err": float(w["repricing_rel_err"].median()),
    })


    # 4) ANCHOR CONSISTENCY: quarter_df vs weekly anchor row
    anchor_w = (
        w.loc[w["source"] == "anchor_em", [
            "quarter_no", "anchor_date", "A_hat", "theta", "PD_Q_1y", "PD_P_1y"
        ]]
        .rename(columns={
            "A_hat": "A_anchor_from_weekly",
            "theta": "theta_anchor_from_weekly",
            "PD_Q_1y": "PD_Q_anchor_from_weekly",
            "PD_P_1y": "PD_P_anchor_from_weekly",
        })
        .copy()
    )

    anchor_cmp = q.merge(anchor_w, on=["quarter_no", "anchor_date"], how="left")
    anchor_cmp["A_anchor_abs_diff"] = np.abs(anchor_cmp["A_anchor"] - anchor_cmp["A_anchor_from_weekly"])
    anchor_cmp["theta_anchor_abs_diff"] = np.abs(anchor_cmp["theta_anchor"] - anchor_cmp["theta_anchor_from_weekly"])
    anchor_cmp["PD_Q_anchor_abs_diff"] = np.abs(anchor_cmp["PD_Q_anchor"] - anchor_cmp["PD_Q_anchor_from_weekly"])
    anchor_cmp["PD_P_anchor_abs_diff"] = np.abs(anchor_cmp["PD_P_anchor"] - anchor_cmp["PD_P_anchor_from_weekly"])

    anchor_checks = pd.Series({
        "max_A_anchor_diff": float(anchor_cmp["A_anchor_abs_diff"].max()),
        "max_theta_anchor_diff": float(anchor_cmp["theta_anchor_abs_diff"].max()),
        "max_PD_Q_anchor_diff": float(anchor_cmp["PD_Q_anchor_abs_diff"].max()),
        "max_PD_P_anchor_diff": float(anchor_cmp["PD_P_anchor_abs_diff"].max()),
    })


    # 5) ECONOMIC DIAGNOSTICS

    w["A_over_L"] = w["A_hat"] / w["L"]
    w["ln_A_over_L"] = np.log(w["A_over_L"])
    w["cds_pd_rough_1y_R40"] = w["cds"].apply(lambda x: rough_pd_from_cds_1y(x, recovery=recovery_for_cds))

    anchor_econ = (
        w.loc[w["source"] == "anchor_em", [
            "quarter_no", "anchor_date", "A_hat", "E", "L", "r", "cds",
            "A_over_L", "ln_A_over_L", "PD_Q_1y", "PD_P_1y", "cds_pd_rough_1y_R40",
            "alpha", "beta1", "delta", "beta0", "theta"
        ]]
        .sort_values("anchor_date")
        .reset_index(drop=True)
    )

    weekly_econ_summary = (
        w.groupby("quarter_no", as_index=False)
         .agg(
             n_weeks=("date", "size"),
             A_over_L_min=("A_over_L", "min"),
             A_over_L_median=("A_over_L", "median"),
             A_over_L_max=("A_over_L", "max"),
             PD_Q_min=("PD_Q_1y", "min"),
             PD_Q_median=("PD_Q_1y", "median"),
             PD_Q_max=("PD_Q_1y", "max"),
             cds_min=("cds", "min"),
             cds_median=("cds", "median"),
             cds_max=("cds", "max"),
         )
         .sort_values("quarter_no")
         .reset_index(drop=True)
    )


    # 6) QUICK FLAGS

    flags = []

    if not shape_checks["all_weekly_dates_are_last_trading_day"]:
        flags.append("Some weekly score dates are not the last available trading day of the week.")

    if not feasibility_checks["all_weekly_pricing_feasible"]:
        flags.append("At least one weekly row violates NIG pricing feasibility.")

    if repricing_checks["max_rel_repricing_err"] > 1e-6:
        flags.append("Weekly inversion does not reprice observed equity closely enough; inspect inversion accuracy.")

    if not repricing_checks["all_Ahat_ge_E"]:
        flags.append("At least one row has A_hat < E, which is economically suspicious for call-option equity.")

    if (anchor_econ["PD_Q_1y"] < 1e-12).all():
        flags.append("All anchor-date Q-PDs are essentially zero; likely need economic diagnostics on leverage / liability proxy / parameter scaling.")

    # output bundle
    out = {
        "shape_checks": shape_checks,
        "feasibility_checks": feasibility_checks,
        "repricing_checks": repricing_checks,
        "anchor_checks": anchor_checks,
        "anchor_compare": anchor_cmp,
        "anchor_econ": anchor_econ,
        "weekly_econ_summary": weekly_econ_summary,
        "weekly_diagnostics": w,
        "flags": flags,
    }
    return out

# RUN SANITY CHECKS

sanity = run_sanity_checks_nig_pilot(
    pilot_quarter_nig_df,
    pilot_weekly_nig_df,
    nig_df_onefirm,
    week_ending="W-FRI",
    recovery_for_cds=0.40,
)

print("=== SHAPE / CALENDAR CHECKS ===")
display(sanity["shape_checks"].to_frame("value"))

print("=== FEASIBILITY CHECKS ===")
display(sanity["feasibility_checks"].to_frame("value"))

print("=== REPRICING CHECKS ===")
display(sanity["repricing_checks"].to_frame("value"))

print("=== ANCHOR CONSISTENCY CHECKS ===")
display(sanity["anchor_checks"].to_frame("value"))

print("=== ANCHOR-LEVEL ECONOMIC DIAGNOSTICS ===")
display(sanity["anchor_econ"])

print("=== QUARTERLY WEEKLY-SUMMARY DIAGNOSTICS ===")
display(sanity["weekly_econ_summary"])

print("=== FLAGS ===")
if len(sanity["flags"]) == 0:
    print("No major red flags triggered by the current sanity checks.")
else:
    for i, flag in enumerate(sanity["flags"], 1):
        print(f"{i}. {flag}")



=== SHAPE / CALENDAR CHECKS ===


,value
n_quarters,4
n_weekly_rows,52
n_unique_weekly_dates,52
n_duplicate_quarter_date_rows,0
all_weekly_dates_are_last_trading_day,True
n_anchor_rows,4
one_anchor_per_quarter,True


=== FEASIBILITY CHECKS ===


,value
all_basic_params_feasible,True
all_weekly_pricing_feasible,True
min_alpha_minus_abs_beta_theta,111.02095
min_alpha_minus_abs_beta_theta_plus1,112.02095


=== REPRICING CHECKS ===


...

=== ANCHOR-LEVEL ECONOMIC DIAGNOSTICS ===


,quarter_no,anchor_date,A_hat,E,L,r,cds,A_over_L,ln_A_over_L,PD_Q_1y,PD_P_1y,cds_pd_rough_1y_R40,alpha,beta1,delta,beta0,theta
0,1,2014-01-03,1.160909e+11,8.335634e+10,3.276700e+10,0.000991,8.05,3.542921,1.264951,2.136974e-18,6.918153e-25,0.001341,150.291186,7.546631,3.003749,0.056639,-10.830443
1,2,2014-04-04,1.122948e+11,8.186783e+10,3.046400e+10,0.001217,6.48,3.686146,1.304582,8.172367e-17,1.310141e-22,0.001079,116.809212,3.671908,2.666617,0.120640,-9.397855
2,3,2014-07-04,1.173817e+11,8.691221e+10,3.046400e+10,-0.000181,6.42,3.853130,1.348886,7.839417e-20,5.732767e-26,0.001069,122.654073,13.692812,2.566542,-0.094174,-9.703994
3,4,2014-10-03,1.202792e+11,8.978999e+10,3.046400e+10,-0.000829,7.21,3.948242,1.373270,2.348812e-19,2.499482e-24,0.001201,116.490598,8.386744,2.556871,-0.024217,-7.821224


=== QUARTERLY WEEKLY-SUMMARY DIAGNOSTICS ===


,quarter_no,n_weeks,A_over_L_min,A_over_L_median,A_over_L_max,PD_Q_min,PD_Q_median,PD_Q_max,cds_min,cds_median,cds_max
0,1,13,3.408536,3.555534,3.694489,1.811917e-19,1.737499e-18,2.009029e-17,6.35,6.82,8.70
1,2,13,3.518244,3.779783,3.879784,6.367046e-18,2.488942e-17,8.257243e-16,5.48,6.49,7.43
2,3,13,3.582311,3.745956,4.047951,4.374008e-21,4.059622e-19,5.173432e-18,5.10,6.42,9.40
3,4,13,3.790013,4.062586,4.283831,2.217350e-21,4.662971e-20,2.239002e-18,4.10,5.68,9.19


=== FLAGS ===
1. All anchor-date Q-PDs are essentially zero; likely need economic diagnostics on leverage / liability proxy / parameter scaling.
